In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import accuracy_score
from sklearn.metrics import mutual_info_score
from sklearn.linear_model import LogisticRegression

In [4]:
df = pd.read_csv("https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv")

(5000, 9)


,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [6]:
print(df.shape)

(5000, 9)


In [7]:
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   lead_source               4852 non-null   object 
 1   industry                  4760 non-null   object 
 2   employment_status         4806 non-null   object 
 3   location                  4792 non-null   object 
 4   annual_income             4631 non-null   float64
 5   number_of_courses_viewed  5000 non-null   int64  
 6   interaction_count         5000 non-null   int64  
 7   lead_score                4965 non-null   float64
 8   converted                 5000 non-null   int64  
dtypes: float64(2), int64(3), object(4)
memory usage: 351.7+ KB


In [9]:
df.isnull().sum()

,0
lead_source,148
industry,240
employment_status,194
location,208
annual_income,369
number_of_courses_viewed,0
interaction_count,0
lead_score,35
converted,0


In [15]:
cat_cols = list(df.select_dtypes(include="object").columns)
num_cols = list(df.select_dtypes(include="number").columns)

print(f"categorical features {cat_cols}")
print(f"numerical features {num_cols}")

categorical features ['lead_source', 'industry', 'employment_status', 'location']
numerical features ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score', 'converted']


In [16]:
df[cat_cols] = df[cat_cols].fillna("NA")
df[num_cols] = df[num_cols].fillna(0.0)

## Question 1
What is the most frequent observation (mode) for the column industry?

In [19]:
print(df["industry"].mode()[0])

technology


## Question 2
Create the correlation matrix for the numerical features of your dataset. In a correlation matrix, you compute the correlation coefficient between every pair of features.

What are the two features that have the biggest correlation?
- interaction_count and lead_score
- number_of_courses_viewed and lead_score
- number_of_courses_viewed and - - interaction_count
- annual_income and interaction_count

In [20]:
corr_matrix = df[num_cols].corr()

corr_matrix.loc[
    ["interaction_count", "number_of_courses_viewed", "annual_income"],
        ["lead_score", "interaction_count", "number_of_courses_viewed"]
        ]

,lead_score,interaction_count,number_of_courses_viewed
interaction_count,0.915746,1.000000,0.721609
number_of_courses_viewed,0.757204,0.721609,1.000000
annual_income,0.229496,0.122842,0.161300


## Question 3
- Calculate the mutual information score between converted and other categorical variables in the dataset. Use the training set only.
- Round the scores to 2 decimals using round(score, 2).
Which of these variables has the biggest mutual information score?

- industry
- location
- lead_source
- employment_status

In [23]:
from sklearn.model_selection import train_test_split
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
    )

In [24]:
from sklearn.metrics import mutual_info_score

def calculate_mi(series):
    return round(
            mutual_info_score(df_train["converted"], series), 2
                )

In [25]:
mi = df_train[cat_cols].apply(calculate_mi)

print(mi.sort_values(ascending=False))

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64


## Question 4
- Now let's train a logistic regression.
- Remember that we have several categorical variables in the dataset. Include them using one-hot encoding.
- Fit the model on the training dataset.
To make sure the results are reproducible across different versions of Scikit-Learn, fit the model with these parameters:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
- Calculate the accuracy on the validation dataset and round it to 2 decimal digits.
What accuracy did you get?

- 0.55
- 0.65
- 0.75
- 0.85

In [26]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

In [28]:
y_train = df_train["converted"]
y_val = df_val["converted"]
X_train_dict = df_train.drop(columns=["converted"]).to_dict(orient="records")
X_val_dict = df_val.drop(columns=["converted"]).to_dict(orient="records")

In [29]:
dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(X_train_dict)
X_val = dv.transform(X_val_dict)

In [31]:
model = LogisticRegression(
    solver="liblinear",
    C=1.0,
    max_iter=1000,
    random_state=42 )
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000, random_state=42, solver='liblinear')

In [36]:
y_pred = model.predict(X_val)
original_accuracy = (y_pred == y_val).mean()
round(original_accuracy, 2)

np.float64(0.64)

## Question 5
- Let's find the least useful feature using the feature elimination technique.
- Train a model using the same features and parameters as in Q4 (without rounding).
- Now exclude each feature from this set and train a model without it. Record the accuracy for each model.
- For each feature, calculate the difference between the original accuracy and the accuracy without the feature.
Which of following feature has the smallest difference?

- 'lead_source'
- 'number_of_courses_viewed'
- 'interaction_count'

In [33]:

col_to_test = [
    'lead_source',
    'number_of_courses_viewed',
    'interaction_count'
]

for col in col_to_test:
    train_dicts_reduced = [
        {k: v for k, v in row.items() if k != col}
        for row in X_train_dict
    ]

    val_dicts_reduced = [
        {k: v for k, v in row.items() if k != col}
        for row in X_val_dict
    ]

    dv_reduced = DictVectorizer(sparse=False)

    X_train_reduced = dv_reduced.fit_transform(
        train_dicts_reduced
    )
    X_val_reduced = dv_reduced.transform(
        val_dicts_reduced
    )

    model_reduced = LogisticRegression(
        solver='liblinear',
        C=1.0,
        max_iter=1000,
        random_state=42
    )

    model_reduced.fit(X_train_reduced, y_train)

    y_pred = model_reduced.predict(X_val_reduced)
    accuracy = (y_pred == y_val).mean()

    difference = original_accuracy - accuracy

    print(
        f"Feature: {col}, "
        f"Accuracy: {accuracy:.4f}, "
        f"Difference: {difference:.4f}"
    )


Feature: lead_source, Accuracy: 0.6420, Difference: 0.0030
Feature: number_of_courses_viewed, Accuracy: 0.6430, Difference: 0.0020
Feature: interaction_count, Accuracy: 0.6010, Difference: 0.0440


## Question 6
- Now let's train a regularized logistic regression.
- Let's try the following values of the parameter C: [0.000001, 0.00001, 0.0001, 0.001].
- Train models using all the features as in Q4.
- Calculate the accuracy on the validation dataset and round it to 3 decimal digits.
Which of these C leads to the best accuracy on the validation set?

- 0.000001
- 0.00001
- 0.0001
- 0.001

In [34]:

C = [0.000001, 0.00001, 0.0001, 0.001]

for i in C:
    model = LogisticRegression(
        solver='liblinear',
        C=i,
        max_iter=1000,
        random_state=42
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)
    accuracy = (y_pred == y_val).mean()

    print(f"C = {i}, Accuracy = {accuracy:.3f}")


C = 1e-06, Accuracy = 0.598
C = 1e-05, Accuracy = 0.598
C = 0.0001, Accuracy = 0.613
C = 0.001, Accuracy = 0.645
